In [1]:
# imports and configuration
import pandas as pd
import requests
import time
from pathlib import Path

## Functions

In [14]:
def load_tournament_results(filepath):
    tournament_results = pd.read_csv(filepath)
    print(f"loaded {len(tournament_results):,} records")
    print(f"unique cards: {tournament_results['card_name'].nunique():,}")
    return tournament_results

### Card weight calculations ###

# apply mainboard/sideboard weighting to card copies
def apply_copy_weights(tournament_results, mainboard_weight, sideboard_weight):
    tournament_results["weighted_copies"] = tournament_results.apply(
        lambda row: row["card_count"] * mainboard_weight if row["is_mainboard"]
        else row["card_count"] * sideboard_weight, axis=1
    )
    return tournament_results

def explore_tournament_size_buckets(tournament_results):
    bucket_counts = tournament_results.groupby("tournament_size_bucket")["source_file"].nunique()
    bucket_records = tournament_results.groupby("tournament_size_bucket")["card_name"].count()
    summary = pd.DataFrame({
        "tournaments": bucket_counts,
        "records": bucket_records
    })
    summary["pct_tournaments"] = (summary["tournaments"] / summary["tournaments"].sum() * 100).round(1)
    summary["pct_records"] = (summary["records"] / summary["records"].sum() * 100).round(1)
    print(summary.to_string())

def apply_tournament_size_weights(tournament_results, tournament_size_weights):
    tournament_results["tournament_size_weight"] = tournament_results["tournament_size_bucket"].apply(#return the value associated with the key that matches the tournament size
        lambda x: tournament_size_weights[x]
    )
    tournament_results["weighted_copies"] = tournament_results["weighted_copies"] * tournament_results["tournament_size_weight"]
    print("applied size weights:")
    for bucket, weight in tournament_size_weights.items():
        count = (tournament_results["tournament_size_bucket"] == bucket).sum()
        #print(f"  {bucket}: {weight} ({count:,} records)")
    return tournament_results

def aggregate_raw_performance(tournament_results):
    card_performance = tournament_results.groupby("card_name").agg(
        total_weighted_copies=("weighted_copies", "sum"),
        deck_appearances=("player", "count"),
        avg_weighted_copies_per_deck=("weighted_copies", "mean"),
        avg_win_rate=("win_rate", "mean"),
    ).reset_index()
    card_performance["avg_weighted_copies_per_deck"] = card_performance["avg_weighted_copies_per_deck"].round(2)
    card_performance["avg_win_rate"] = card_performance["avg_win_rate"].round(2)

    #print(f"aggregated {len(card_performance):,} unique cards")
    return card_performance

def calculate_card_scores(card_performance):
    card_performance["performance_score"] = card_performance["total_weighted_copies"] * card_performance["avg_win_rate"]
    card_scores = card_performance.sort_values("performance_score", ascending=False).reset_index(drop=True)
    #print(f"calculated scores for {len(card_scores):,} unique cards")
    return card_scores

### Misc ###
def show_cards(df, card_names):
    mask = df["card_name"].isin(card_names)
    return(df[mask].to_string(index=False))

## Code

### Universal Variables

In [3]:
data_dir = Path("C:/Users/david/Documents/GitHub/ai_projects/modern_cube/data")
filtered_data = data_dir / "processed/filtered_card_records_2016_2018.csv"
output_dir = data_dir / "processed"

# cube configuration
cube_size = 360

# weights for metrics
#As we do not have a way to know how often a sideboard card is used we have to guestimate
mainboard_weight = 1
sideboard_weight = .5

#this is based on distribution of tournaments in the data which is show later on
tournament_size_weights = {
    "small": 0.25,
    "medium": 1.0,
    "large": 2.0,
    "major": 4.0
}

### Executed Code

In [4]:
tournament_results = load_tournament_results(filtered_data)
tournament_results.head()

loaded 118,070 records
unique cards: 1,633


,card_name,card_count,is_mainboard,player,player_wins,player_losses,tournament_name,tournament_date,source_file,tournament_size,record,win_rate,tournament_size_bucket
0,Ancient Stirrings,4,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0,small
1,Chromatic Sphere,4,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0,small
2,Chromatic Star,4,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0,small
3,"Emrakul, the Aeons Torn",1,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0,small
4,Expedition Map,4,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0,small


In [5]:
tournament_results = apply_copy_weights(tournament_results, mainboard_weight, sideboard_weight)
tournament_results.head()

,card_name,card_count,is_mainboard,player,player_wins,player_losses,tournament_name,tournament_date,source_file,tournament_size,record,win_rate,tournament_size_bucket,weighted_copies
0,Ancient Stirrings,4,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0,small,4.0
1,Chromatic Sphere,4,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0,small,4.0
2,Chromatic Star,4,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0,small,4.0
3,"Emrakul, the Aeons Torn",1,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0,small,1.0
4,Expedition Map,4,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0,small,4.0


In [6]:
# use this to determine weight of tournaments
explore_tournament_size_buckets(tournament_results)

                        tournaments  records  pct_tournaments  pct_records
tournament_size_bucket                                                    
large                           106    28197             22.2         23.9
major                            23    16312              4.8         13.8
medium                          142    36594             29.7         31.0
small                           207    36967             43.3         31.3


In [11]:
tournament_results = apply_tournament_size_weights(tournament_results, tournament_size_weights)

applied size weights:
  small: 0.25 (36,967 records)
  medium: 1.0 (36,594 records)
  large: 2.0 (28,197 records)
  major: 4.0 (16,312 records)


In [12]:
tournament_results.head()

,card_name,card_count,is_mainboard,player,player_wins,player_losses,tournament_name,tournament_date,source_file,tournament_size,record,win_rate,tournament_size_bucket,weighted_copies,tournament_size_weights,tournament_size_weight
0,Ancient Stirrings,4,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0,small,1.00,0.25,0.25
1,Chromatic Sphere,4,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0,small,1.00,0.25,0.25
2,Chromatic Star,4,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0,small,1.00,0.25,0.25
3,"Emrakul, the Aeons Torn",1,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0,small,0.25,0.25,0.25
4,Expedition Map,4,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0,small,1.00,0.25,0.25


In [ ]:
card_performance = aggregate_raw_performance(tournament_results)
card_scores = calculate_card_scores(card_performance)

In [ ]:
 # TODO on scryfall get modern legal date

In [40]:
card_performance = aggregate_raw_performance(tournament_results)
card_performance.head()

,card_name,total_weighted_copies,deck_appearances,avg_weighted_copies_per_deck,avg_win_rate
0,Abbot of Keral Keep,22.5,10,2.25,0.80
1,Abrade,100.5,104,0.97,0.79
2,Abrupt Decay,1108.0,569,1.95,0.81
3,Abundant Growth,7.0,3,2.33,0.85
4,Abzan Ascendancy,4.0,1,4.00,0.80


In [60]:
banned_cards = ["Splinter Twin", "Summer Bloom", "Eye of Ugin", 'Gitaxian Probe', 'Golgari Grave-Troll']
show_cards(card_scores, banned_cards).head()

,card_name,total_weighted_copies,deck_appearances,avg_weighted_copies_per_deck,avg_win_rate,performance_score
24,Eye of Ugin,1715.0,493,3.48,0.81,1389.15
76,Gitaxian Probe,971.0,252,3.85,0.82,796.22
319,Golgari Grave-Troll,177.0,45,3.93,0.85,150.45
332,Summer Bloom,172.0,43,4.00,0.80,137.60
345,Splinter Twin,164.0,42,3.90,0.79,129.56


In [59]:
unbanned_cards = ['Ancestral Vision', 'Sword of the Meek', 'Jace, the Mind Sculptor', 'Bloodbraid Elf']
show_cards(card_scores, unbanned_cards).head()

,card_name,total_weighted_copies,deck_appearances,avg_weighted_copies_per_deck,avg_win_rate,performance_score
297,Ancestral Vision,215.0,88,2.44,0.80,172.00
313,"Jace, the Mind Sculptor",199.5,88,2.27,0.78,155.61
424,Bloodbraid Elf,115.0,30,3.83,0.78,89.70
1041,Sword of the Meek,6.0,3,2.00,0.80,4.80


In [41]:
card_scores = calculate_card_scores(card_performance)
card_scores.head()

,card_name,total_weighted_copies,deck_appearances,avg_weighted_copies_per_deck,avg_win_rate,performance_score
0,Lightning Bolt,5011.0,1482,3.38,0.81,4058.910
1,Island,4105.5,1362,3.01,0.80,3284.400
2,Path to Exile,3793.0,1332,2.85,0.81,3072.330
3,Mountain,3496.5,1646,2.12,0.81,2832.165
4,Wooded Foothills,3460.5,1119,3.09,0.81,2803.005
